# Exercise 06 — aggregating consumer & anomaly detection

**Goal:** build a consumer that doesn't just *print* events but **processes** them in real time:

1. Parse the JSON payload.
2. Maintain running statistics per house and sensor (count, min,    max, average).
3. Flag values above a threshold as anomalies.
4. Print a summary table — and optionally update it live.

**Prerequisite:** run [`exercise_05_produce_stream.ipynb`](exercise_05_produce_stream.ipynb) (especially Task B — anomaly simulation) so there is data to aggregate.

## Background — state in a streaming context

In batch processing you load all data, then aggregate. In streaming you maintain *state* that updates as each event arrives. Our state lives in a few `defaultdict`s — that's enough for a single-process consumer.

Real production stream processors (Flink, Kafka Streams, …) replicate and persist this state so it survives crashes. We're keeping it in RAM for clarity.

In [ ]:
from confluent_kafka import Consumer
from collections import defaultdict
from datetime import datetime
import json

# Anything above these is treated as an anomaly.
THRESHOLDS = {
    'strom':  50.0,    # kWh per reading
    'wasser': 200.0,   # Liter per reading
}

consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          'aggregation-exercise',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe(['strom', 'wasser'])
print('Consumer ready.')

## Step 1 — read and aggregate (up to 200 events)

Three accumulators:

- `totals[topic][house]` — sum of values (used for the average).
- `counts[topic][house]` — number of events (denominator).
- `anomalies` — list of events that exceeded the threshold.

**Task:** complete the aggregation inside the loop.

In [ ]:
totals    = defaultdict(lambda: defaultdict(float))
counts    = defaultdict(lambda: defaultdict(int))
anomalies = []

messages_read, empty_polls = 0, 0
while messages_read < 200 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1

    try:
        data  = json.loads(msg.value().decode())
        house = data.get('haus', 'unknown')
        topic = msg.topic()
        value = float(data.get('wert', 0.0))

        # TODO: update totals[topic][house] and counts[topic][house]


        # TODO: if value > THRESHOLDS[topic], append to anomalies:
        #   anomalies.append({'topic': topic, 'house': house,
        #                     'value': value, 'offset': msg.offset()})

    except Exception as e:
        print(f'Parse error: {e}')

print(f'Done — processed {messages_read} events.')

## Step 2 — print the summary table

Format:

```
Topic   | House   | Count |    Avg
------------------------------------
strom   | haus_a  |    12 |  7.34 kWh
```

**Task:** loop over `totals`, compute `avg = totals[t][h] / counts[t][h]`, and print one row per `(topic, house)` pair. Use `'kWh'` for `strom` and `'L'` for `wasser`.

In [ ]:
# TODO: print a summary table


## Step 3 — anomaly report

List all detected anomalies. Show how far above the threshold each is, in percent. Hint: `pct_over = (value / THRESHOLDS[topic] - 1) * 100`.

If there are no anomalies, re-run the anomaly simulator in `exercise_05_produce_stream.ipynb` Task B and try again.

In [ ]:
# TODO: print each anomaly with topic, house, value and percent over threshold


## Bonus — live aggregation

Build a continuous loop that prints an updated summary every 10 events. While it runs, fire the streaming producer in another tab and watch the averages converge.

Tip: `from IPython.display import clear_output; clear_output(wait=True)` lets you redraw the same area instead of scrolling forever.

In [ ]:
# TODO: live aggregation loop


In [ ]:
try:
    consumer.close()
except Exception:
    pass
print('Closed.')

## What you learned

- A *processing* consumer maintains in-memory state and updates it   per message — fundamentally different from "just print".
- Threshold-based anomaly detection is dead simple but surprisingly   effective for monotonic sensors. Statistical methods (z-score,   IQR) come later.
- This whole pattern — "per-key running aggregate" — is exactly   what dedicated stream processors like Flink were built for. We'll   meet Flink in the next track.